In [ ]:
from openai import OpenAI
from anthropic import Anthropic
from dotenv import load_dotenv
import os
import csv
import pandas as pd
import json

# Load the environment variables
load_dotenv()

# Initialize the OpenAI client
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [ ]:
def parse_job_postings(job_content: str, job_title: str, company: str):
    prompt = f"""Parse this job posting and extract metadata in JSON format.

    Job Title: {job_title}
    Company: {company}
    Job Description: {job_content}

    Extract the following information and return ONLY valid JSON (no markdown, no backticks, no explanation):

    {{
        "location_type": "remote" | "hybrid" | "onsite" | unknown,
        "location": "city, state/country" or null,
        "salary_min": number or null,
        "salary_max": number or null,
        "salary_currency": "USD" | "EUR" | "GBP" etc or null,
        "department": "Tech" | "Finance" | "Marketing" | "Sales" | "Engineering" | "Design" | "Customer Service" | "Other" etc or null,
        "job_type": "full-time" | "part-time" | "contract" | "temporary" | "volunteer" | "internship" | "other" or null,
        "industry": "Tech" | "Finance" | "Marketing" | "Sales" | "Engineering" | "Design" | "Other" etc or null
    }}

    Rules:
    1. For location_type, look for keywords: "remote", "hybrid", "onsite", "in-office", "work from home"
    2. Extract salary even if it's a range. Convert to numbers (no $ or commas)
    3. If information is not clearly stated, use null.
    4. DO NOT include any text outside the JSON object
    5. DO NOT use markdown code blocks or backticks
    """

    try:
        message = anthropic_client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4000,
            messages=[{"role": "user", "content": prompt}]
        )

        response_text = message.content[0].text.strip()
        
        # Check if response is empty
        if not response_text:
            print(f"Empty response from API for job: {job_title}")
            return None
        
        # Try to extract JSON if it's wrapped in markdown code blocks
        if response_text.startswith("```"):
            # Remove markdown code blocks
            lines = response_text.split("\n")
            response_text = "\n".join([line for line in lines if not line.strip().startswith("```")])
            response_text = response_text.strip()
        
        metadata = json.loads(response_text)

        # print(metadata)
        return metadata

    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        try:
            print(f"Response text: {response_text[:200]}...")  # Print first 200 chars for debugging
        except:
            print("Response text not available")
        return None

    except Exception as e:
        print(f"Error parsing job: {e}")
        return None

In [ ]:
# Read CSV, parse jobs, and write to new CSV with metadata
output_filename = "job_postings_with_metadata.csv"

with open("job_postings.csv", "r", encoding="utf-8") as infile, \
     open(output_filename, "w", newline="", encoding="utf-8") as outfile:
    
    reader = csv.DictReader(infile)
    
    # Define fieldnames: existing columns + metadata columns
    fieldnames = [
        "company", "absolute_url", "title", "content", "fetched_date",
        "location_type", "location", "salary_min", "salary_max", "salary_currency",
        "department", "job_type", "industry"
    ]
    
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()
    
    for idx, row in enumerate(reader, 1):
        company = row.get("company", "").strip()
        job_title = row.get("title", "").strip()
        job_content = row.get("content", "").strip()
        
        # Stop processing if row is empty (no more data)
        if not company or not job_title or not job_content:
            print(f"Reached empty row at {idx}. Stopping processing.")
            break
        
        print(f"Processing job {idx}: {job_title} at {company}")
        metadata = parse_job_postings(job_content, job_title, company)
        
        # Combine existing row data with metadata
        output_row = {
            "company": row["company"],
            "absolute_url": row["absolute_url"],
            "title": row["title"],
            "content": row["content"],
            "fetched_date": row["fetched_date"],
            "location_type": metadata.get("location_type") if metadata else None,
            "location": metadata.get("location") if metadata else None,
            "salary_min": metadata.get("salary_min") if metadata else None,
            "salary_max": metadata.get("salary_max") if metadata else None,
            "salary_currency": metadata.get("salary_currency") if metadata else None,
            "department": metadata.get("department") if metadata else None,
            "job_type": metadata.get("job_type") if metadata else None,
            "industry": metadata.get("industry") if metadata else None,
        }
        
        writer.writerow(output_row)
    
    print(f"\nCompleted! Wrote results to {output_filename}")

Processing job 1: AirCover UX Enablement Manager at airbnb
Processing job 2: Business Operations Associate at airbnb
Processing job 3: Claims Experience Specialist (French-Speaking) at airbnb
Processing job 4: Community Engagement Intern, Homes at airbnb
Processing job 5: Community Support Senior Business Operations Lead at airbnb
Processing job 6: Consumer Insights Lead, China at airbnb
Processing job 7: Consumer & Product Communications Lead, APAC at airbnb
Processing job 8: Creative Production Lead, Marcom at airbnb
Processing job 9: CS Delivery Manager at airbnb
Processing job 10: CS Labs Ambassador at airbnb
Processing job 11: Data Science Lead, Guest Funnel Science at airbnb
Processing job 12: Engineering Manager, CS Data Services at airbnb
Processing job 13: Engineering Manager, Data Frameworks at airbnb
Processing job 14: Engineering Manager, Host Pricing - Product UI Foundations at airbnb
Processing job 15: Engineering Manager, Network Infrastructure (Cloud) at airbnb
Processi

KeyboardInterrupt: 